## Load/Prep Data

In [1]:
import sys
!{sys.executable} -m pip install grn-dazzle scanpy scipy scikit-learn

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 25.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ----------------- ---------------------- 15.7/36.5 MB 77.9 MB/s eta 0:00:01
   ---------------------------------- ----- 31.5/36.5 MB 77.5 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 67.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 8.0/8.0 MB 66.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 66.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 8.1/8.1 MB 64.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------------------------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import scanpy as sc
from scipy.stats import pearsonr
from sklearn.preprocessing import minmax_scale
from dazzle import runDAZZLE, DEFAULT_DAZZLE_CONFIGS
import scipy.sparse as sp

In [3]:
# 1. Load Metadata and Expression Data
print("Loading data...")
metadata = pd.read_csv('Data/CHOOSE-multiome-wt-metadata.csv', index_col=0)

# Pandas can read .csv.gz directly
expr_data = pd.read_csv('Data/CHOOSE-multiome-wt-log-norm.csv.gz', index_col=0) 

# Ensure the expression matrix columns match metadata rows
expr_data = expr_data[metadata.index]

# Convert to an AnnData object for easier slicing by cell type
adata = sc.AnnData(X=expr_data.T)
adata.obs['celltype_jf'] = metadata['celltype_jf']

print(f"Loaded {adata.n_obs} cells and {adata.n_vars} genes.")

Loading data...
Loaded 732 cells and 15685 genes.


## Run DAZZLE for each cell type (going off of GRNBoost2)

In [ ]:
# 1. Target TFs from CHOOSE-tf-to-analyze-metadata.csv
# Please verify these symbols against your local CSV file!
target_tfs = [
    'BCL11A', 'TCF4', 'ARID1B', 'ADNP', 'ASH1L', 'CHD8', 
    'DYRK1A', 'POGZ', 'SHANK3', 'SYNGAP1', 'TBR1', 
    'SCN2A', 'GRIN2B', 'FOXP1'
]

all_predictions = pd.DataFrame()

# Loop through each cell type in your metadata
for cell_type in adata.obs['celltype_jf'].unique():
    print(f"--- Processing cell type: {cell_type} ---")
    
    # Subset to current cell type
    adata_ct = adata[adata.obs['celltype_jf'] == cell_type].copy()
    
    # 2. Filter out all-zero genes (DAZZLE requirement)
    sc.pp.filter_genes(adata_ct, min_cells=1)
    print(f"  -> {adata_ct.n_vars} active genes remaining.")
    
    # Convert to dense array for DAZZLE
    expr_array = adata_ct.X.toarray() if sp.issparse(adata_ct.X) else adata_ct.X
        
    # 3. Run DAZZLE
    print("  -> Training DAZZLE model (this may take a few minutes)...")
    model, adjs = runDAZZLE(expr_array, DEFAULT_DAZZLE_CONFIGS)
    
    # 4. Extract results
    adj_matrix = model.get_adj()
    gene_names = np.array(adata_ct.var_names)
    
    # --- OPTIMIZATION: Only process our 14 TFs ---
    # Find which of our target TFs are actually present in this cell type
    present_tfs = [gene for gene in target_tfs if gene in gene_names]
    tf_indices = [np.where(gene_names == gene)[0][0] for gene in present_tfs]
    
    # Slice the matrix: [rows_for_TFs, all_target_columns]
    reduced_adj = adj_matrix[tf_indices, :] 
    
    # Create the DataFrame (14 rows x ~15,000 columns)
    adj_df = pd.DataFrame(reduced_adj, index=present_tfs, columns=gene_names)
    
    # Melt into 3-column format: TF, target, importance
    dazzle_edges = adj_df.reset_index().melt(
        id_vars='index', 
        var_name='target', 
        value_name='importance'
    )
    dazzle_edges.rename(columns={'index': 'TF'}, inplace=True)
    
    # Add cell type label and append
    dazzle_edges['cell.type'] = cell_type
    all_predictions = pd.concat([all_predictions, dazzle_edges], ignore_index=True)

print("\nSuccess! All cell types processed.")

# Post process - calculating the Pearson correlation between the TF and the target gene.

In [ ]:
def calculate_mor_and_format(predictions_df, adata_obj):
    """
    Translates the R post-processing logic to Python.
    Calculates Pearson correlation to determine the sign (+ or -) of the edge,
    then scales the final score to [-1, 1].
    """
    final_results = []
    
    # Group by cell type so we calculate correlations within the correct cell population
    for cell_type, group in predictions_df.groupby('cell.type'):
        # Get the expression matrix for this cell type
        ct_expr = adata_obj[adata_obj.obs['celltype_jf'] == cell_type].to_df()
        
        for _, row in group.iterrows():
            tf = row['TF']
            target = row['target']
            importance = row['importance']
            
            # Skip if TF or target isn't in our expression matrix
            if tf not in ct_expr.columns or target not in ct_expr.columns:
                continue
                
            # 1. Calculate Pearson correlation between TF and target expression
            # This gives us the direction of regulation (positive = induce, negative = repress)
            tf_expr = ct_expr[tf].values
            target_expr = ct_expr[target].values
            
            # Note: if a gene has zero variance (e.g., all 0s), correlation will be NaN
            if np.std(tf_expr) == 0 or np.std(target_expr) == 0:
                corr = 0
            else:
                corr, _ = pearsonr(tf_expr, target_expr)
            
            # 2. Multiply DAZZLE's non-negative importance by the sign of the correlation
            raw_score = importance * np.sign(corr)
            
            final_results.append({
                'TF': tf,
                'target': target,
                'cell.type': cell_type,
                'raw_score': raw_score,
                'importance': importance # Keep for p-value calculation if needed
            })
            
    res_df = pd.DataFrame(final_results)
    
    # 3. Scale the scores so the maximum absolute value is 1 (matches the R script)
    # all.res[,"score"] <- all.res[,"score"] / max(abs(all.res[,"score"]))
    max_abs_score = res_df['raw_score'].abs().max()
    res_df['score'] = res_df['raw_score'] / max_abs_score
    
    return res_df[['TF', 'cell.type', 'target', 'score']]

# Run the post-processing
final_submission = calculate_mor_and_format(all_predictions, adata)

# Save your final predictions!
final_submission.to_csv('dazzle_predictions_submission.csv', index=False)
print("Ready to submit!")